<a href="https://colab.research.google.com/github/c-marq/cap4767-data-mining/blob/main/demos/week03_PredictingInsuranceCosts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# Setup: Libraries and Data Loading
# Prerequisites: None — all libraries pre-installed in Colab
# ============================================

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_regression
from sklearn.metrics import (mean_squared_error, r2_score,
                             classification_report, confusion_matrix,
                             ConfusionMatrixDisplay)

# Load the dataset
url = 'https://raw.githubusercontent.com/c-marq/CAP3321C-Data-Wrangling/refs/heads/main/data/insurance.csv'
insurance = pd.read_csv(url)

print("Insurance data:", insurance.shape)
print(insurance.head())
print(f"\nNull values: {insurance.isnull().sum().sum()}")

In [ ]:
# The most important EDA finding in the dataset
print("--- Charges by Smoker Status ---")
print(insurance.groupby('smoker')['charges'].agg(['mean', 'median', 'count']).round(2))

In [ ]:
# ============================================
# Visualize Key Relationships
# ============================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age vs Charges (colored by smoker status)
colors = insurance['smoker'].map({'yes': 'red', 'no': 'steelblue'})
axes[0].scatter(insurance['age'], insurance['charges'], c=colors, alpha=0.4, s=15)
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Charges ($)')
axes[0].set_title('Age vs Charges (Red = Smoker)')

# BMI vs Charges (colored by smoker status)
axes[1].scatter(insurance['bmi'], insurance['charges'], c=colors, alpha=0.4, s=15)
axes[1].set_xlabel('BMI')
axes[1].set_ylabel('Charges ($)')
axes[1].set_title('BMI vs Charges (Red = Smoker)')

# Distribution of charges
axes[2].hist(insurance['charges'], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[2].set_xlabel('Charges ($)')
axes[2].set_ylabel('Count')
axes[2].set_title('Distribution of Insurance Charges')
axes[2].axvline(x=insurance['charges'].median(), color='red', linestyle='--',
                label=f"Median: ${insurance['charges'].median():,.0f}")
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# Simple Linear Regression: age → charges
# ============================================

correlation = insurance['age'].corr(insurance['charges'])
print(f"Correlation between age and charges: {correlation:.3f}")

X_train, X_test, y_train, y_test = train_test_split(
    insurance[['age']],       # Features — must be 2D (DataFrame, not Series)
    insurance['charges'],     # Target
    test_size=0.33,
    random_state=42
)

model_simple = LinearRegression()
model_simple.fit(X_train, y_train)

r2 = model_simple.score(X_test, y_test)
y_pred = model_simple.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² on test set: {r2:.3f}")
print(f"RMSE: ${rmse:,.2f}")
print(f"Slope (per year of age): ${model_simple.coef_[0]:,.2f}")
print(f"Intercept: ${model_simple.intercept_:,.2f}")

In [ ]:
# ============================================
# Visualize the Simple Regression Line + Residuals
# ============================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Scatter with regression line
axes[0].scatter(X_test, y_test, alpha=0.3, s=10, color='steelblue')
x_range = np.linspace(X_test['age'].min(), X_test['age'].max(), 100).reshape(-1, 1)
axes[0].plot(x_range, model_simple.predict(x_range), color='red', linewidth=2)
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Charges ($)')
axes[0].set_title(f'Simple Regression: age → charges (R² = {r2:.3f})')

# Residual plot
residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.3, s=10, color='steelblue')
axes[1].axhline(y=0, color='red', linestyle='--')
axes[1].set_xlabel('Predicted Charges ($)')
axes[1].set_ylabel('Residual ($)')
axes[1].set_title('Residual Plot — Simple Regression')

# Predicted vs Actual
axes[2].scatter(y_test, y_pred, alpha=0.3, s=10, color='steelblue')
axes[2].plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()],
             'r--', label='Perfect prediction')
axes[2].set_xlabel('Actual Charges ($)')
axes[2].set_ylabel('Predicted Charges ($)')
axes[2].set_title('Predicted vs Actual')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# Create dummy variables and build progressive models
# ============================================

insurance_encoded = insurance.copy()
insurance_encoded['smoker_yes'] = (insurance_encoded['smoker'] == 'yes').astype(int)
insurance_encoded['sex_male'] = (insurance_encoded['sex'] == 'male').astype(int)
region_dummies = pd.get_dummies(insurance_encoded['region'], prefix='region', dtype=int)
insurance_encoded = insurance_encoded.join(region_dummies)
insurance_encoded = insurance_encoded.drop(columns=['sex', 'smoker', 'region'])

# Model A: age only
X_tr, X_te, y_tr, y_te = train_test_split(
    insurance_encoded[['age']], insurance_encoded['charges'],
    test_size=0.25, random_state=42
)
m = LinearRegression().fit(X_tr, y_tr)
print(f"Model A (age only):               R² = {m.score(X_te, y_te):.3f}")

# Model B: age + bmi
X_tr, X_te, y_tr, y_te = train_test_split(
    insurance_encoded[['age', 'bmi']], insurance_encoded['charges'],
    test_size=0.25, random_state=42
)
m = LinearRegression().fit(X_tr, y_tr)
print(f"Model B (age + bmi):              R² = {m.score(X_te, y_te):.3f}")

# Model C: age + bmi + children
X_tr, X_te, y_tr, y_te = train_test_split(
    insurance_encoded[['age', 'bmi', 'children']], insurance_encoded['charges'],
    test_size=0.25, random_state=42
)
m = LinearRegression().fit(X_tr, y_tr)
print(f"Model C (age + bmi + children):   R² = {m.score(X_te, y_te):.3f}")

# Model D: age + bmi + children + smoker (THE BIG JUMP)
X_tr, X_te, y_tr, y_te = train_test_split(
    insurance_encoded[['age', 'bmi', 'children', 'smoker_yes']],
    insurance_encoded['charges'], test_size=0.25, random_state=42
)
model_d = LinearRegression().fit(X_tr, y_tr)
r2_d = model_d.score(X_te, y_te)
y_pred_d = model_d.predict(X_te)
rmse_d = np.sqrt(mean_squared_error(y_te, y_pred_d))
print(f"Model D (+ smoker):               R² = {r2_d:.3f}, RMSE = ${rmse_d:,.2f}")

# Model E: all features including sex and region dummies
feature_cols_e = ['age', 'bmi', 'children', 'smoker_yes', 'sex_male',
                  'region_northwest', 'region_southeast', 'region_southwest']

X_tr, X_te, y_tr, y_te = train_test_split(
    insurance_encoded[feature_cols_e], insurance_encoded['charges'],
    test_size=0.25, random_state=42
)
model_e = LinearRegression().fit(X_tr, y_tr)
r2_e = model_e.score(X_te, y_te)
y_pred_e = model_e.predict(X_te)
rmse_e = np.sqrt(mean_squared_error(y_te, y_pred_e))
print(f"Model E (all features):           R² = {r2_e:.3f}, RMSE = ${rmse_e:,.2f}")

In [ ]:
# Coefficient analysis — what each feature contributes
coef_df = pd.DataFrame({
    'Feature': feature_cols_e,
    'Coefficient': model_e.coef_
}).sort_values(by='Coefficient', ascending=False)
print(coef_df.to_string(index=False))
print(f"\nIntercept: ${model_e.intercept_:,.2f}")

In [ ]:
# ============================================
# Residual Analysis — Multiple Regression Model
# ============================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

residuals_e = y_te - y_pred_e

# Residual scatter plot
axes[0].scatter(y_pred_e, residuals_e, alpha=0.3, s=10, color='steelblue')
axes[0].axhline(y=0, color='red', linestyle='--')
axes[0].set_xlabel('Predicted Charges ($)')
axes[0].set_ylabel('Residual ($)')
axes[0].set_title(f'Residual Plot — Multiple Regression (R² = {r2_e:.3f})')

# Predicted vs Actual scatter
axes[1].scatter(y_te, y_pred_e, alpha=0.3, s=10, color='steelblue')
axes[1].plot([y_te.min(), y_te.max()],
             [y_te.min(), y_te.max()],
             'r--', label='Perfect prediction')
axes[1].set_xlabel('Actual Charges ($)')
axes[1].set_ylabel('Predicted Charges ($)')
axes[1].set_title('Predicted vs Actual — Multiple Regression')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# Feature Selection: test every value of k
# ============================================

all_feature_cols = ['age', 'bmi', 'children', 'smoker_yes', 'sex_male',
                    'region_northeast', 'region_northwest',
                    'region_southeast', 'region_southwest']

X = insurance_encoded[all_feature_cols]
y = insurance_encoded['charges']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)

train_scores, test_scores = [], []
for k in range(1, len(all_feature_cols) + 1):
    fs = SelectKBest(score_func=mutual_info_regression, k=k)
    fs.fit(X_tr, y_tr)
    X_tr_sel = fs.transform(X_tr)
    X_te_sel = fs.transform(X_te)
    temp_model = LinearRegression().fit(X_tr_sel, y_tr)
    train_scores.append(temp_model.score(X_tr_sel, y_tr))
    test_scores.append(temp_model.score(X_te_sel, y_te))

results = pd.DataFrame({
    'k': range(1, len(all_feature_cols) + 1),
    'Train R²': train_scores, 'Test R²': test_scores
})
results.plot(x='k', y=['Train R²', 'Test R²'], figsize=(10, 5), marker='o')
plt.xlabel('Number of Features (k)')
plt.ylabel('R² Score')
plt.title('Feature Selection: Finding the Sweet Spot')
plt.xticks(range(1, len(all_feature_cols) + 1))
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ============================================
# Model Comparison Bar Chart
# ============================================

models = ['A: age', 'B: age+bmi', 'C: +children', 'D: +smoker', 'E: all features']
r2_values = [0.099, 0.129, 0.127, 0.765, 0.767]

plt.figure(figsize=(10, 5))
bars = plt.bar(models, r2_values, color=['#ccc', '#ccc', '#ccc', '#00BFA5', '#00897B'],
               edgecolor='white', linewidth=1.5)
plt.ylabel('R² (Test Set)')
plt.title('Progressive Model Improvement: One Feature Changes Everything')
plt.ylim(0, 1.0)

# Add value labels on bars
for bar, val in zip(bars, r2_values):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
             f'{val:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# Logistic Regression: Predicting High-Cost Patients
# ============================================

# Create binary target: charges > $15,000
threshold = 15000
insurance_encoded['high_cost'] = (insurance_encoded['charges'] > threshold).astype(int)

print(f"Threshold: ${threshold:,}")
print(f"High-cost rate: {insurance_encoded['high_cost'].mean():.1%}")
print(f"\nClass distribution:")
print(insurance_encoded['high_cost'].value_counts().rename({0: 'Standard Cost', 1: 'High Cost'}))

# Explore high-cost patterns
print(f"\n--- High-cost rate by smoker status ---")
print(insurance.groupby('smoker').apply(
    lambda x: (x['charges'] > threshold).mean()).round(3).rename('high_cost_rate'))

In [ ]:
# ============================================
# Logistic Regression: Classify high-cost patients
# ============================================

# Create binary target
insurance_encoded['high_cost'] = (insurance_encoded['charges'] > 15000).astype(int)
print(f"High-cost rate: {insurance_encoded['high_cost'].mean():.1%}")

# Define features and split
log_features = ['age', 'bmi', 'children', 'smoker_yes', 'sex_male']
X_log = insurance_encoded[log_features]
y_log = insurance_encoded['high_cost']

X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(
    X_log, y_log, test_size=0.25, random_state=42
)

# Scale features for logistic regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_log)
X_test_scaled = scaler.transform(X_test_log)

# Fit — same .fit() call, different model
log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train_scaled, y_train_log)

# Evaluate
y_pred_log = log_model.predict(X_test_scaled)
print(f"Accuracy: {log_model.score(X_test_scaled, y_test_log):.3f}")
print("\nClassification Report:")
print(classification_report(y_test_log, y_pred_log,
                            target_names=['Standard Cost', 'High Cost']))

# Confusion matrix
cm = confusion_matrix(y_test_log, y_pred_log)
disp = ConfusionMatrixDisplay(cm, display_labels=['Standard Cost', 'High Cost'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — High-Cost Patient Classifier')
plt.show()

In [ ]:
# ============================================
# Logistic Regression Coefficients + Probabilities
# ============================================

print("Logistic Regression Coefficients:")
coef_log_df = pd.DataFrame({
    'Feature': log_features,
    'Coefficient': log_model.coef_[0]
}).sort_values(by='Coefficient', ascending=False)
print(coef_log_df.to_string(index=False))

# Show predicted probabilities for first 10 test cases
probabilities = log_model.predict_proba(X_test_scaled)[:10]
print(f"\nSample predicted probabilities (first 10 test cases):")
print(f"{'Prob Standard':>15} {'Prob High-Cost':>15} {'Prediction':>12} {'Actual':>8}")
for i in range(10):
    pred = y_pred_log[i] if hasattr(y_pred_log, '__getitem__') else list(y_pred_log)[i]
    actual = y_test_log.iloc[i]
    print(f"{probabilities[i][0]:>15.3f} {probabilities[i][1]:>15.3f} {pred:>12} {actual:>8}")